# 01 — Ground classification + Z-normalisation (Site 01)

This notebook is a **reproducible, modular** workflow for:

1. Listing LAS/LAZ tiles  
2. Ground classification (choose **CSF** or **SMRF**)  
3. Height-above-ground / Z-normalisation (writes `Z_norm` as an extra dimension)  
4. Quick QC plots (class counts, HAG histogram, 2D/top-view and 3D views)  
5. Optional batch processing with multiprocessing  

Design patterns are inspired by:
- https://github.com/trchudley/pdemtools
- https://github.com/trchudley/crevdem
- https://github.com/mdtanker/RIS_basement_sediment

> Recommended workflow: run **Setup → Config → List files → Single-tile test → QC → Batch**.


## Setup

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Literal, Optional, Tuple, List
import os
import json
import math
import logging

import numpy as np

# Optional (used only in plotting/QC cells)
import matplotlib.pyplot as plt

In [2]:
# ---- Optional dependencies ----
# The workflow can run with:
#   - PDAL (recommended) for ground classification + HAG
#   - laspy (for quick inspection / reading attributes)
#
# If PDAL isn't available in your environment, you can still run QC on outputs,
# but classification/HAG steps will be skipped with a clear message.

try:
    import pdal  # type: ignore
    pdal_available = True
except Exception as e:
    pdal_available = False
    print("⚠️ PDAL python bindings not available:", repr(e))

try:
    import laspy  # type: ignore
    laspy_available = True
except Exception as e:
    laspy_available = False
    print("⚠️ laspy not available:", repr(e))

try:
    from tqdm.auto import tqdm  # type: ignore
    tqdm_available = True
except Exception:
    tqdm_available = False
    tqdm = None  # type: ignore

## Configuration

In [3]:
@dataclass(frozen=True)
class GroundConfig:
    # --- Site / IO ---
    site_name: str = "site1_DD0001"
    input_dir: Path = Path("/data/calperumResearch/site1_1_DD0001/lidar/clipped/raster/coreg/icp")
    output_base: Path = Path("/data/calperumResearch/site1_1_DD0001/lidar/clipped/raster")
    pattern: str = "tiles_multispectral*.las"  # also supports *.laz if you change it

    # --- Processing ---
    method: Literal["csf", "smrf"] = "csf"
    compute_hag: bool = True  # adds HeightAboveGround, then writes as Z_norm
    overwrite: bool = False

    # --- CSF params (PDAL filters.csf) ---
    csf_rigidness: int = 3
    csf_iterations: int = 500
    csf_time_step: float = 0.65
    csf_resolution: float = 0.5
    csf_threshold: float = 0.01

    # --- SMRF params (PDAL filters.smrf) ---
    smrf_slope: float = 0.15
    smrf_window: float = 3.5
    smrf_threshold: float = 0.25
    smrf_scalar: float = 1.35

    # --- Parallel ---
    n_workers: int = 4

    # --- Outputs ---
    @property
    def out_ground_dir(self) -> Path:
        return self.output_base / "ground_classification" / self.method

    @property
    def out_znorm_dir(self) -> Path:
        return self.output_base / "z_norm" / self.method


CFG = GroundConfig()

# Create output folders
CFG.out_ground_dir.mkdir(parents=True, exist_ok=True)
CFG.out_znorm_dir.mkdir(parents=True, exist_ok=True)

print("✅ Config loaded")
print("Site:", CFG.site_name)
print("Input:", CFG.input_dir)
print("Outputs:", CFG.output_base)
print("Method:", CFG.method)
print("Pattern:", CFG.pattern)

✅ Config loaded
Site: site1_DD0001
Input: /data/calperumResearch/site1_1_DD0001/lidar/clipped/raster/coreg/icp
Outputs: /data/calperumResearch/site1_1_DD0001/lidar/clipped/raster
Method: csf
Pattern: tiles_multispectral*.las


## List input tiles

In [4]:
las_files = list_las_files(CFG.input_dir, CFG.pattern)

print(f"Found {len(las_files)} LAS/LAZ tiles")
if las_files:
    print("First 5:")
    for f in las_files[:5]:
        print(" -", f.name)

NameError: name 'list_las_files' is not defined

## Core processing functions

In [5]:
def list_las_files(input_dir: Path, pattern: str) -> List[Path]:
    files = sorted(input_dir.glob(pattern))
    return [f for f in files if f.suffix.lower() in {".las", ".laz"}]

def _pdal_pipeline(json_obj: dict):
    if not pdal_available:
        raise RuntimeError("PDAL python bindings are not available in this environment.")
    return pdal.Pipeline(json.dumps(json_obj))

def classify_ground_pdal(
    in_las: Path,
    out_las: Path,
    method: Literal["csf", "smrf"],
    cfg: GroundConfig,
) -> None:
    '''
    Classify ground using PDAL. Writes output LAS with updated Classification.
    '''
    if not pdal_available:
        raise RuntimeError("PDAL is required for ground classification steps.")

    if out_las.exists() and not cfg.overwrite:
        return

    filters = []
    if method == "csf":
        filters.append({
            "type": "filters.csf",
            "rigidness": cfg.csf_rigidness,
            "iterations": cfg.csf_iterations,
            "time_step": cfg.csf_time_step,
            "resolution": cfg.csf_resolution,
            "threshold": cfg.csf_threshold,
            "classify": True,  # sets Classification=2 for ground
        })
    elif method == "smrf":
        filters.append({
            "type": "filters.smrf",
            "slope": cfg.smrf_slope,
            "window": cfg.smrf_window,
            "threshold": cfg.smrf_threshold,
            "scalar": cfg.smrf_scalar,
        })
    else:
        raise ValueError(f"Unknown method: {method}")

    pipeline = {
        "pipeline": [
            str(in_las),
            *filters,
            {"type": "writers.las", "filename": str(out_las), "extra_dims": "all"},
        ]
    }
    pipe = _pdal_pipeline(pipeline)
    pipe.execute()

def add_z_norm_pdal(in_ground_las: Path, out_znorm_las: Path, cfg: GroundConfig) -> None:
    '''
    Compute HeightAboveGround (HAG) and store it as an extra dimension `Z_norm`.
    Uses PDAL HAG filter(s) which operate on ground-classified points.
    '''
    if not pdal_available:
        raise RuntimeError("PDAL is required for HAG/Z_norm steps.")

    if out_znorm_las.exists() and not cfg.overwrite:
        return

    pipeline = {
        "pipeline": [
            str(in_ground_las),
            {"type": "filters.hag_delaunay"},
            {"type": "filters.ferry", "dimensions": "HeightAboveGround=>Z_norm"},
            {"type": "writers.las", "filename": str(out_znorm_las), "extra_dims": "all"},
        ]
    }
    pipe = _pdal_pipeline(pipeline)
    pipe.execute()

def process_one_tile(tile: Path, cfg: GroundConfig) -> Tuple[Path, Optional[Path]]:
    '''
    Process one tile: ground classification -> optional Z_norm.
    Returns (ground_out, znorm_out or None).
    '''
    ground_out = cfg.out_ground_dir / f"ground_{cfg.method}_{tile.name}"
    znorm_out = cfg.out_znorm_dir / f"z_norm_{cfg.method}_{tile.name}"

    classify_ground_pdal(tile, ground_out, cfg.method, cfg)

    if cfg.compute_hag:
        add_z_norm_pdal(ground_out, znorm_out, cfg)
        return ground_out, znorm_out

    return ground_out, None

## Single-tile test

In [6]:
# ---- Single-tile test (recommended) ----
if not las_files:
    raise SystemExit("No LAS/LAZ files found - check CFG.input_dir and CFG.pattern")

test_tile = las_files[0]
print("Test tile:", test_tile)

if not pdal_available:
    print("⚠️ PDAL not available -> skipping processing. Install PDAL or run on an env with pdal-python.")
else:
    ground_out, znorm_out = process_one_tile(test_tile, CFG)
    print("✅ Ground out:", ground_out)
    print("✅ Z_norm out:", znorm_out)

NameError: name 'las_files' is not defined

## QC helpers

In [ ]:
def read_las(path: Path):
    if not laspy_available:
        raise RuntimeError("laspy is required for QC reading.")
    return laspy.read(path)

def classification_counts(las) -> dict:
    cls = np.asarray(las.classification, dtype=np.uint8)
    unique, counts = np.unique(cls, return_counts=True)
    return {int(k): int(v) for k, v in zip(unique, counts)}

def get_dim(las, name: str) -> np.ndarray:
    if not hasattr(las, name):
        raise AttributeError(f"LAS has no dimension '{name}'. Available: {list(las.point_format.dimension_names)}")
    return np.asarray(getattr(las, name))

def quick_hag_hist(z_norm: np.ndarray, title: str = "Z_norm (HAG) histogram", bins: int = 100):
    z = z_norm[np.isfinite(z_norm)]
    plt.figure(figsize=(10, 4))
    plt.hist(z, bins=bins)
    plt.title(title)
    plt.xlabel("metres")
    plt.ylabel("count")
    plt.grid(True, alpha=0.3)
    plt.show()

## QC on outputs

In [ ]:
# ---- QC on outputs from the single-tile run ----
if laspy_available and las_files and pdal_available:
    ground_out = CFG.out_ground_dir / f"ground_{CFG.method}_{las_files[0].name}"
    znorm_out = CFG.out_znorm_dir / f"z_norm_{CFG.method}_{las_files[0].name}"

    if ground_out.exists():
        las_g = read_las(ground_out)
        counts = classification_counts(las_g)
        print("Classification counts:", counts)
        if 2 in counts:
            print(f"Ground (class=2): {counts[2]:,} points ({counts[2]/len(las_g.points)*100:.2f}%)")
        else:
            print("⚠️ No class=2 ground points found - check parameters.")

    if znorm_out.exists():
        las_z = read_las(znorm_out)
        if "Z_norm" in las_z.point_format.dimension_names:
            z_norm = get_dim(las_z, "Z_norm").astype(np.float32)
            print("Z_norm stats:", float(np.nanmin(z_norm)), float(np.nanmean(z_norm)), float(np.nanmax(z_norm)))
            quick_hag_hist(z_norm, title=f"{CFG.method.upper()} Z_norm histogram")
        else:
            print("⚠️ Output LAS missing Z_norm dimension. Check PDAL ferry step / writer.")

## Batch processing

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def _process_one_for_pool(tile_str: str, cfg_dict: dict) -> Tuple[str, str]:
    # Recreate config inside subprocess (dataclasses aren't trivially pickle-safe across envs)
    cfg = GroundConfig(**cfg_dict)
    tile = Path(tile_str)
    ground_out, znorm_out = process_one_tile(tile, cfg)
    return str(ground_out), str(znorm_out) if znorm_out else ""

def process_batch(cfg: GroundConfig, max_tiles: Optional[int] = None) -> None:
    files = list_las_files(cfg.input_dir, cfg.pattern)
    if max_tiles is not None:
        files = files[:max_tiles]
    if not files:
        print("No files to process.")
        return
    if not pdal_available:
        print("⚠️ PDAL not available -> cannot batch process.")
        return

    cfg_dict = cfg.__dict__.copy()
    print(f"Batch processing {len(files)} tiles with {cfg.n_workers} workers...")

    iterable = files
    if tqdm_available:
        iterable = tqdm(files, desc="Submitting", unit="tile")  # type: ignore

    futures = []
    with ProcessPoolExecutor(max_workers=cfg.n_workers) as ex:
        for f in iterable:
            futures.append(ex.submit(_process_one_for_pool, str(f), cfg_dict))

        done_iter = as_completed(futures)
        if tqdm_available:
            done_iter = tqdm(done_iter, total=len(futures), desc="Processing", unit="tile")  # type: ignore

        for fut in done_iter:
            try:
                ground_out, znorm_out = fut.result()
            except Exception as e:
                print("❌ Failed:", repr(e))
            else:
                pass

    print("✅ Batch complete")

In [ ]:
# ---- Batch run (uncomment to execute) ----
# process_batch(CFG, max_tiles=None)

## Interactive file selection (optional)

If you have `ipywidgets` installed, this lets you pick an output LAS and visualise `Z_norm` quickly.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    widgets_available = True
except Exception as e:
    widgets_available = False
    print("⚠️ ipywidgets not available:", repr(e))

if widgets_available:
    # Build a file list from outputs (prefer Z_norm files)
    out_files = sorted(CFG.out_znorm_dir.glob("z_norm_*.*"))
    if not out_files:
        out_files = sorted(CFG.out_ground_dir.glob("ground_*.*"))

    dropdown = widgets.Dropdown(
        options=[(p.name, str(p)) for p in out_files],
        description="LAS file:",
        layout=widgets.Layout(width="90%"),
    )
    display(dropdown)

In [ ]:
def plot_top_view(las_path: Path, use_dim: str = "Z_norm", n: int = 200_000):
    if not laspy_available:
        raise RuntimeError("laspy required for plotting.")
    las = laspy.read(las_path)
    x = np.asarray(las.x)
    y = np.asarray(las.y)

    if use_dim in las.point_format.dimension_names:
        v = np.asarray(getattr(las, use_dim), dtype=np.float32)
    else:
        v = np.asarray(las.z, dtype=np.float32)

    # Downsample for speed
    if len(x) > n:
        idx = np.random.choice(len(x), n, replace=False)
        x, y, v = x[idx], y[idx], v[idx]

    plt.figure(figsize=(8, 7))
    plt.scatter(x, y, s=0.1, c=v)
    plt.title(f"{las_path.name} — coloured by {use_dim if use_dim in las.point_format.dimension_names else 'Z'}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.2)
    plt.show()

if widgets_available:
    def _on_change(change):
        if change["name"] == "value" and change["new"]:
            plot_top_view(Path(change["new"]), use_dim="Z_norm")
    dropdown.observe(_on_change, names="value")

## 3D visualisation (PyVista, optional)

If you have PyVista installed, you can render a point cloud interactively.


In [ ]:
try:
    import pyvista as pv
    pyvista_available = True
except Exception as e:
    pyvista_available = False
    print("⚠️ PyVista not available:", repr(e))

if pyvista_available and laspy_available:
    # Pick the first Z_norm output if it exists
    z_files = sorted(CFG.out_znorm_dir.glob("z_norm_*.las"))
    if z_files:
        p = z_files[0]
        las = laspy.read(p)
        pts = np.vstack([np.asarray(las.x), np.asarray(las.y), np.asarray(las.z)]).T
        vals = np.asarray(las.Z_norm) if "Z_norm" in las.point_format.dimension_names else np.asarray(las.z)

        cloud = pv.PolyData(pts)
        cloud["val"] = vals.astype(np.float32)

        pl = pv.Plotter()
        pl.add_points(cloud, scalars="val", point_size=2, render_points_as_spheres=True)
        pl.add_axes()
        pl.show_grid()
        pl.show()
    else:
        print("No z_norm outputs found yet.")

## Notes

- If you already have a `groundnorm` module, you can swap the core PDAL functions above with your existing wrappers.
- For SMRF vs CSF, start with your current defaults and tune on a few representative tiles.
- For strict reproducibility, record your PDAL version and parameter values in your thesis/paper methods.